# Divisão e conquista: multiplicação de matrizes de Strassen — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As medições e os gráficos ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- contar as operações do **algoritmo clássico** e confirmar experimentalmente o custo $\Theta(n^3)$;
- implementar o **produto por blocos** e ver por que $T(n) = 8T(n/2) + \Theta(n^2)$ continua sendo $\Theta(n^3)$;
- implementar as **7 multiplicações de Strassen** e verificar que o número de produtos escalares é $7^k = n^{\log_2 7}$;
- escolher um **ponto de corte** entre Strassen e o algoritmo clássico, primeiro contando operações e depois medindo tempo;
- representar um algoritmo de multiplicação como uma **decomposição de tensor**, a ideia por trás do AlphaTensor.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True)
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída dos programas deste tutorial."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            return float(tok.split('=')[1])
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

def matriz(saida):
    """Lê as linhas numéricas da saída (as que não contêm '=') como lista de listas."""
    return [[float(x) for x in lin.split()] for lin in saida.splitlines()
            if lin.strip() and '=' not in lin]

# Exemplo condutor dos slides: n, depois A e B linha a linha.
EXEMPLO = '2\n1 3\n7 5\n6 8\n4 2\n'
C_EXEMPLO = [[18, 14], [62, 66]]

## 1. O algoritmo clássico

Cada entrada $c_{ij} = \sum_{k} a_{ik} b_{kj}$ é o produto da linha $i$ de $A$ pela coluna $j$
de $B$. São $n^2$ entradas com $n$ multiplicações cada: $n^3$ multiplicações e $n^2(n-1)$ somas.

O programa abaixo é o mesmo dos slides. Ele funciona de dois jeitos:

- `./classico < entrada` lê $n$, $A$ e $B$ e imprime $C$;
- `./classico n semente` sorteia matrizes $n \times n$ de inteiros pequenos e mede o tempo.

Dobrar $n$ deve multiplicar o tempo por $2^3 = 8$.

**Fonte:** Cormen et al. (2009), §4.2; Levitin (2012), §2.3.

In [ ]:
%%writefile src/classico.c
/* PRODUTO DE MATRIZES PELO ALGORITMO CLASSICO: tres lacos, n^3 multiplicacoes.
   Uso: ./classico < entrada     (n, A e B na entrada; imprime C)
        ./classico n semente     (matrizes aleatorias; mede o tempo)
   Cormen et al. (2009), sec. 4.2. */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

static long long mults = 0;

/* C = A B, matrizes n x n em ordem de linhas */
static void multClassico( int n, const double *A, const double *B, double *C) {
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) {
         double s = 0.0;
         for (int k = 0; k < n; ++k)
            s += A[i*n+k] * B[k*n+j];
         C[i*n+j] = s;
      }
   mults += (long long) n * n * n;
}

int main( int argc, char *argv[]) {
   int n;
   if (argc == 1) { if (scanf( "%d", &n) != 1) return 1; }
   else n = atoi( argv[1]);
   double *A = malloc( (size_t) n*n * sizeof (double));
   double *B = malloc( (size_t) n*n * sizeof (double));
   double *C = malloc( (size_t) n*n * sizeof (double));
   if (argc == 1) {
      for (int i = 0; i < 2*n*n; ++i)
         if (scanf( "%lf", i < n*n ? &A[i] : &B[i-n*n]) != 1) return 1;
      multClassico( n, A, B, C);
      for (int i = 0; i < n; ++i, putchar( '\n'))
         for (int j = 0; j < n; ++j) printf( "%g ", C[i*n+j]);
      printf( "mults=%lld\n", mults);
   } else {
      srand( atoi( argv[2]));
      for (int i = 0; i < n*n; ++i) { A[i] = rand() % 11 - 5; B[i] = rand() % 11 - 5; }
      clock_t t0 = clock();
      multClassico( n, A, B, C);
      double t = (double) (clock() - t0) / CLOCKS_PER_SEC;
      printf( "n=%d mults=%lld tempo=%.4f\n", n, mults, t);
   }
   free( A); free( B); free( C);
   return 0;
}

In [ ]:
# Exemplo: o 2x2 dos slides e a assinatura de Theta(n^3) no tempo.
saida = rodar('classico.c', entrada=EXEMPLO)
assert matriz(saida) == C_EXEMPLO and campo(saida, 'mults') == 8

tempos_cl = {}
for n in [128, 256, 512, 1024]:
    s = rodar('classico.c', args=(n, 1))
    tempos_cl[n] = campo(s, 'tempo')
ns = sorted(tempos_cl)
for a, b in zip(ns, ns[1:]):
    print(f'T({b})/T({a}) = {tempos_cl[b] / max(tempos_cl[a], 1e-9):.1f}   (esperado ~8)')

In [ ]:
# Exercício 1: contando as operações do algoritmo clássico
# TODO: devolva a tupla (multiplicacoes, somas) que o algoritmo clássico faz
# para multiplicar duas matrizes n x n. Lembre que somar n parcelas exige n - 1 somas.

def contar_classico(n):
    # TODO: implemente aqui
    pass

# Teste automático
assert contar_classico(1) == (1, 0), 'n = 1: uma multiplicação, nenhuma soma'
assert contar_classico(2) == (8, 4), 'n = 2: 8 multiplicações e 4 somas'
assert contar_classico(10) == (1000, 900), 'Verifique sua implementação'
print('Exercício 1 ok! Total de operações para n = 1000:', sum(contar_classico(1000)))

## 2. Divisão e conquista ingênua: produto por blocos

Com $n$ potência de 2, partimos cada matriz em quatro blocos $\frac n2 \times \frac n2$ e usamos
a regra do produto $2\times 2$ com blocos no lugar de números:

$$C_{11} = A_{11}B_{11} + A_{12}B_{21}, \quad C_{12} = A_{11}B_{12} + A_{12}B_{22}, \quad
  C_{21} = A_{21}B_{11} + A_{22}B_{21}, \quad C_{22} = A_{21}B_{12} + A_{22}B_{22}.$$

São **8** produtos recursivos: $T(n) = 8T(n/2) + \Theta(n^2) = \Theta(n^{\log_2 8}) = \Theta(n^3)$.

**Detalhe de implementação.** Para não copiar blocos, o tipo `Mat` guarda um ponteiro para o
canto superior esquerdo e a largura da linha da matriz original (`ld`, *leading dimension*). Um
bloco é só um ponteiro deslocado com o mesmo `ld`. É assim que bibliotecas como a BLAS
representam submatrizes.

**Fonte:** Cormen et al. (2009), §4.2.

In [ ]:
%%writefile src/blocos.c
/* PRODUTO POR BLOCOS (divisao e conquista ingenua): 8 produtos de n/2.
   Uso: ./blocos n semente     (n potencia de 2; confere com o classico)
   Cormen et al. (2009), sec. 4.2, SQUARE-MATRIX-MULTIPLY-RECURSIVE. */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

typedef struct { double *p; int ld; } Mat;     /* elemento (i,j) em p[i*ld + j] */
#define EL(M,i,j) ((M).p[(i)*(M).ld + (j)])

static long long mults = 0;

static Mat novaMat( int n) { Mat R = { calloc( (size_t) n*n, sizeof (double)), n }; return R; }
static Mat bloco( Mat M, int h, int bi, int bj) { Mat R = { M.p + bi*h*M.ld + bj*h, M.ld }; return R; }

/* Z = X + Y (n x n) */
static void soma( int n, Mat X, Mat Y, Mat Z) {
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) EL(Z,i,j) = EL(X,i,j) + EL(Y,i,j);
}

/* C = A B por blocos; n potencia de 2 */
static void multBlocos( int n, Mat A, Mat B, Mat C) {
   if (n == 1) { EL(C,0,0) = EL(A,0,0) * EL(B,0,0); ++mults; return; }
   int h = n / 2;
   Mat P = novaMat( h), Q = novaMat( h);
   for (int i = 0; i < 2; ++i)                  /* bloco C_ij */
      for (int j = 0; j < 2; ++j) {
         multBlocos( h, bloco( A,h,i,0), bloco( B,h,0,j), P);   /* A_i1 B_1j */
         multBlocos( h, bloco( A,h,i,1), bloco( B,h,1,j), Q);   /* A_i2 B_2j */
         soma( h, P, Q, bloco( C,h,i,j));
      }
   free( P.p); free( Q.p);
}

int main( int argc, char *argv[]) {
   if (argc < 3) return 1;
   int n = atoi( argv[1]);
   srand( atoi( argv[2]));
   Mat A = novaMat( n), B = novaMat( n), C = novaMat( n);
   for (int i = 0; i < n*n; ++i) { A.p[i] = rand() % 11 - 5; B.p[i] = rand() % 11 - 5; }
   multBlocos( n, A, B, C);
   double erro = 0.0;                           /* confere com o classico */
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) {
         double s = 0.0;
         for (int k = 0; k < n; ++k) s += EL(A,i,k) * EL(B,k,j);
         erro = fmax( erro, fabs( s - EL(C,i,j)));
      }
   printf( "n=%d mults=%lld erro=%g\n", n, mults, erro);
   return 0;
}

In [ ]:
# Exemplo: o produto por blocos acerta e faz exatamente n^3 multiplicações.
for n in [1, 2, 4, 8, 16, 32, 64]:
    s = rodar('blocos.c', args=(n, 7))
    assert campo(s, 'erro') == 0 and campo(s, 'mults') == n ** 3
print('Divisão em blocos: mesmas n^3 multiplicações do clássico.')

In [ ]:
# Exercício 2: a recorrência do produto por blocos
# TODO: implemente mult_blocos(n), o número de multiplicações escalares do produto
# por blocos, DIRETAMENTE pela recorrência M(1) = 1, M(n) = 8 M(n/2).
# Depois, implemente custo_nivel(n, j), o custo do nível j da árvore de recursão
# quando cada nó de tamanho m custa m**2 (as somas): são 8**j nós de tamanho n / 2**j.

def mult_blocos(n):
    # TODO: implemente aqui (recursivo)
    pass

def custo_nivel(n, j):
    # TODO: implemente aqui (uma linha)
    pass

# Teste automático
assert all(mult_blocos(2 ** k) == 8 ** k for k in range(11)), 'M(2^k) deve ser 8^k = n^3'
assert custo_nivel(1024, 0) == 1024 ** 2, 'O nível 0 é a raiz: n^2'
assert all(custo_nivel(1024, j + 1) == 2 * custo_nivel(1024, j) for j in range(9)), \
    'Cada nível deve custar o DOBRO do anterior (8 nós, cada um com 1/4 do custo)'
print('Exercício 2 ok! As folhas dominam: a soma dos níveis é Theta(n^3).')

## 3. O algoritmo de Strassen

Strassen (1969) calcula o produto $2\times 2$ com **7** multiplicações:

| produto | | combinação |
|---|---|---|
| $M_1 = (A_{11}+A_{22})(B_{11}+B_{22})$ | | $C_{11} = M_1 + M_4 - M_5 + M_7$ |
| $M_2 = (A_{21}+A_{22})\,B_{11}$ | | $C_{12} = M_3 + M_5$ |
| $M_3 = A_{11}\,(B_{12}-B_{22})$ | | $C_{21} = M_2 + M_4$ |
| $M_4 = A_{22}\,(B_{21}-B_{11})$ | | $C_{22} = M_1 - M_2 + M_3 + M_6$ |
| $M_5 = (A_{11}+A_{12})\,B_{22}$ | | |
| $M_6 = (A_{21}-A_{11})(B_{11}+B_{12})$ | | |
| $M_7 = (A_{12}-A_{22})(B_{21}+B_{22})$ | | |

Aplicada a blocos: $T(n) = 7T(n/2) + \Theta(n^2) = \Theta(n^{\log_2 7}) \approx \Theta(n^{2{,}807})$.

O programa abaixo tem um parâmetro `corte`: quando $n \le$ `corte`, usa o algoritmo clássico.
Com `corte = 1` temos o Strassen "puro", que recursa até $1\times 1$.

- `./strassen < entrada` lê $n$, $A$ e $B$ e imprime $C$ (com `corte = 1`);
- `./strassen n semente corte [verifica]` sorteia as matrizes, mede o tempo e, se `verifica = 1`,
  confere o resultado com o algoritmo clássico.

Como as entradas são inteiras, o resultado tem de ser **exato** (`erro=0`).

**Fonte:** Strassen (1969); Levitin (2012), §5.4; Cormen et al. (2009), §4.2.

In [ ]:
%%writefile src/strassen.c
/* MULTIPLICACAO DE MATRIZES DE STRASSEN com ponto de corte para o classico.
   Uso: ./strassen < entrada                     (n potencia de 2; imprime C; corte = 1)
        ./strassen n semente corte [verifica]    (matrizes aleatorias; mede o tempo)
   Strassen (1969); Cormen et al. (2009), sec. 4.2. */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

typedef struct { double *p; int ld; } Mat;     /* elemento (i,j) em p[i*ld + j] */
#define EL(M,i,j) ((M).p[(i)*(M).ld + (j)])

static long long mults = 0;

static Mat novaMat( int n) { Mat R = { calloc( (size_t) n*n, sizeof (double)), n }; return R; }
static Mat bloco( Mat M, int h, int bi, int bj) { Mat R = { M.p + bi*h*M.ld + bj*h, M.ld }; return R; }

static void soma( int n, Mat X, Mat Y, Mat Z) {        /* Z = X + Y */
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) EL(Z,i,j) = EL(X,i,j) + EL(Y,i,j);
}
static void subtrai( int n, Mat X, Mat Y, Mat Z) {     /* Z = X - Y */
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) EL(Z,i,j) = EL(X,i,j) - EL(Y,i,j);
}

static void classico( int n, Mat A, Mat B, Mat C) {
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) {
         double s = 0.0;
         for (int k = 0; k < n; ++k) s += EL(A,i,k) * EL(B,k,j);
         EL(C,i,j) = s;
      }
   mults += (long long) n * n * n;
}

/* C = A B; n potencia de 2 */
static void strassen( int n, Mat A, Mat B, Mat C, int corte) {
   if (n <= corte) { classico( n, A, B, C); return; }
   int h = n / 2;
   Mat A11 = bloco( A,h,0,0), A12 = bloco( A,h,0,1), A21 = bloco( A,h,1,0), A22 = bloco( A,h,1,1);
   Mat B11 = bloco( B,h,0,0), B12 = bloco( B,h,0,1), B21 = bloco( B,h,1,0), B22 = bloco( B,h,1,1);
   Mat C11 = bloco( C,h,0,0), C12 = bloco( C,h,0,1), C21 = bloco( C,h,1,0), C22 = bloco( C,h,1,1);
   Mat M[8], T1 = novaMat( h), T2 = novaMat( h);
   for (int k = 1; k <= 7; ++k) M[k] = novaMat( h);

   soma( h, A11, A22, T1);    soma( h, B11, B22, T2);    strassen( h, T1,  T2,  M[1], corte);
   soma( h, A21, A22, T1);                               strassen( h, T1,  B11, M[2], corte);
   subtrai( h, B12, B22, T2);                            strassen( h, A11, T2,  M[3], corte);
   subtrai( h, B21, B11, T2);                            strassen( h, A22, T2,  M[4], corte);
   soma( h, A11, A12, T1);                               strassen( h, T1,  B22, M[5], corte);
   subtrai( h, A21, A11, T1); soma( h, B11, B12, T2);    strassen( h, T1,  T2,  M[6], corte);
   subtrai( h, A12, A22, T1); soma( h, B21, B22, T2);    strassen( h, T1,  T2,  M[7], corte);

   for (int i = 0; i < h; ++i)
      for (int j = 0; j < h; ++j) {
         EL(C11,i,j) = EL(M[1],i,j) + EL(M[4],i,j) - EL(M[5],i,j) + EL(M[7],i,j);
         EL(C12,i,j) = EL(M[3],i,j) + EL(M[5],i,j);
         EL(C21,i,j) = EL(M[2],i,j) + EL(M[4],i,j);
         EL(C22,i,j) = EL(M[1],i,j) - EL(M[2],i,j) + EL(M[3],i,j) + EL(M[6],i,j);
      }
   for (int k = 1; k <= 7; ++k) free( M[k].p);
   free( T1.p); free( T2.p);
}

int main( int argc, char *argv[]) {
   int n, corte = 1, verifica = 0;
   if (argc == 1) { if (scanf( "%d", &n) != 1) return 1; }
   else if (argc >= 4) { n = atoi( argv[1]); srand( atoi( argv[2])); corte = atoi( argv[3]);
                         if (argc >= 5) verifica = atoi( argv[4]); }
   else return 1;
   Mat A = novaMat( n), B = novaMat( n), C = novaMat( n);
   if (argc == 1) {
      for (int i = 0; i < 2*n*n; ++i)
         if (scanf( "%lf", i < n*n ? &A.p[i] : &B.p[i-n*n]) != 1) return 1;
   } else
      for (int i = 0; i < n*n; ++i) { A.p[i] = rand() % 11 - 5; B.p[i] = rand() % 11 - 5; }

   clock_t t0 = clock();
   strassen( n, A, B, C, corte);
   double t = (double) (clock() - t0) / CLOCKS_PER_SEC;

   if (argc == 1) {
      for (int i = 0; i < n; ++i, putchar( '\n'))
         for (int j = 0; j < n; ++j) printf( "%g ", EL(C,i,j));
      printf( "mults=%lld\n", mults);
   } else {
      long long m = mults;
      double erro = 0.0;
      if (verifica) {
         Mat D = novaMat( n);
         classico( n, A, B, D);
         for (int i = 0; i < n*n; ++i) erro = fmax( erro, fabs( C.p[i] - D.p[i]));
         free( D.p);
      }
      printf( "n=%d corte=%d mults=%lld tempo=%.4f erro=%g\n", n, corte, m, t, erro);
   }
   return 0;
}

In [ ]:
# Exemplo: o 2x2 dos slides com 7 multiplicações, e 7^k multiplicações em geral.
saida = rodar('strassen.c', entrada=EXEMPLO)
assert matriz(saida) == C_EXEMPLO and campo(saida, 'mults') == 7

mults_st = {}
for k in range(0, 8):
    n = 2 ** k
    s = rodar('strassen.c', args=(n, 3, 1, 1), mostrar=False)
    assert campo(s, 'erro') == 0, f'Strassen errou para n = {n}'
    mults_st[n] = int(campo(s, 'mults'))
    print(f'n={n:4d}  Strassen={mults_st[n]:>10,d}  7^k={7 ** k:>10,d}  clássico n^3={n ** 3:>10,d}')

In [ ]:
# Visualização: multiplicações escalares, clássico x Strassen (escala log-log).
import math
import matplotlib.pyplot as plt

ns = sorted(mults_st)
plt.figure(figsize=(6, 4))
plt.loglog(ns, [n ** 3 for n in ns], 'o-', base=2, label=r'clássico / blocos: $n^3$')
plt.loglog(ns, [mults_st[n] for n in ns], 's-', base=2, label=r'Strassen: $7^k = n^{\log_2 7}$')
plt.xlabel('n')
plt.ylabel('multiplicações escalares')
plt.title(f'Inclinações: 3 e log2(7) = {math.log2(7):.3f}')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

In [ ]:
%%writefile src/strassen2x2.c
/* Exercicio 3: AS 7 MULTIPLICACOES DE STRASSEN (caso escalar 2x2).
   TODO: complete strassen2x2() para calcular c = a b usando EXATAMENTE 7 multiplicacoes
   (use a funcao mul() para cada uma, assim elas sao contadas) e somas/subtracoes livres. */
#include <stdio.h>
#include <stdlib.h>

static int mults = 0;
static long mul( long x, long y) { ++mults; return x * y; }

static void strassen2x2( long a[2][2], long b[2][2], long c[2][2]) {
   /* TODO: calcule m1, ..., m7 com mul() */

   /* TODO: combine-os em c[0][0], c[0][1], c[1][0], c[1][1] */

}

int main( void) {
   long a[2][2] = { {1, 3}, {7, 5} }, b[2][2] = { {6, 8}, {4, 2} }, c[2][2] = { {0} };
   strassen2x2( a, b, c);
   printf( "c=[%ld %ld; %ld %ld] mults=%d\n", c[0][0], c[0][1], c[1][0], c[1][1], mults);
   int ok = (c[0][0] == 18 && c[0][1] == 14 && c[1][0] == 62 && c[1][1] == 66 && mults == 7);
   srand( 2026);
   for (int t = 0; t < 1000 && ok; ++t) {          /* compara com o classico */
      for (int i = 0; i < 2; ++i)
         for (int j = 0; j < 2; ++j) { a[i][j] = rand() % 201 - 100; b[i][j] = rand() % 201 - 100; }
      mults = 0;
      strassen2x2( a, b, c);
      for (int i = 0; i < 2; ++i)
         for (int j = 0; j < 2; ++j)
            if (c[i][j] != a[i][0]*b[0][j] + a[i][1]*b[1][j]) ok = 0;
      if (mults != 7) ok = 0;
   }
   printf( "ok=%d\n", ok);
   return 0;
}

In [ ]:
# Teste automático do Exercício 3
saida = rodar('strassen2x2.c')
assert campo(saida, 'ok') == 1, 'Verifique sua implementação: resultado errado ou número de mul() diferente de 7'
print('Exercício 3 ok! Agora confira à mão C21 = M2 + M4 e C22 = M1 - M2 + M3 + M6 (exercício dos slides).')

## 4. Strassen na prática: o ponto de corte

As 18 somas de Strassen custam caro em matrizes pequenas. Contando **todas** as operações
aritméticas (multiplicações + somas):

- clássico: $2n^3 - n^2$;
- híbrido com corte $n_0$: se $n \le n_0$, usa o clássico; senão faz 7 chamadas de tamanho
  $n/2$ e $18$ somas de matrizes $\frac n2 \times \frac n2$, ou seja, $18\,(n/2)^2$ operações.

Nos slides vimos que um nível de Strassen só compensa quando os blocos têm ordem $m > 7{,}5$.
No exercício abaixo você confirma isso pela contagem. Depois medimos o **tempo real** e
comparamos com a previsão.

Um detalhe importa na comparação: o nosso clássico (ordem `i-j-k`) percorre `B` por **colunas**
e desperdiça o cache em matrizes grandes. Strassen trabalha em blocos menores, que cabem no
cache, e por isso ganha também por **localidade**, não só pela contagem. Diante de um clássico
otimizado (como o da BLAS), o melhor corte sobe para a casa das centenas.

**Fonte:** Higham (2002), cap. 23; Cormen et al. (2009), §4.2.

In [ ]:
# Exercício 4: contando as operações do Strassen híbrido
# TODO: implemente ops_hibrido(n, corte), o total de operações aritméticas
# (multiplicações + somas) do Strassen com ponto de corte, para n potência de 2:
#   - se n <= corte: o custo do clássico, 2n^3 - n^2;
#   - senão: 7 * ops_hibrido(n/2, corte) + 18 * (n/2)^2.

def ops_classico(n):
    return 2 * n ** 3 - n ** 2

def ops_hibrido(n, corte):
    # TODO: implemente aqui (recursivo)
    pass

# Teste automático
assert ops_hibrido(64, 64) == ops_classico(64), 'corte >= n: é o próprio clássico'
assert ops_hibrido(2, 1) == 7 * 1 + 18 * 1 == 25, 'Strassen 2x2 puro: 7 mult. + 18 somas'
assert ops_hibrido(16, 8) < ops_classico(16), 'Um nível sobre blocos 8x8 já compensa (m > 7,5)'
assert ops_hibrido(14 * 2, 14) < ops_classico(28) and ops_hibrido(8, 4) > ops_classico(8), \
    'Verifique sua implementação'

n = 1024
melhor = min((ops_hibrido(n, 2 ** k), 2 ** k) for k in range(11))
print(f'n = {n}: melhor corte pela contagem = {melhor[1]}; '
      f'operações = {melhor[0]:,d} contra {ops_classico(n):,d} do clássico '
      f'({ops_classico(n) / melhor[0]:.2f}x menos)')

In [ ]:
# Exemplo: o tempo real em função do corte (n = 1024).
# Atenção: leva alguns segundos. O melhor corte depende da sua máquina e do compilador.
n = 1024
s = rodar('strassen.c', args=(256, 5, 32, 1))      # conferência rápida da versão com corte
assert campo(s, 'erro') == 0

cortes = [16, 32, 64, 128, 256, 512, 1024]          # corte = n é o próprio clássico
tempos_corte = {}
for c in cortes:
    s = rodar('strassen.c', args=(n, 5, c), mostrar=False)
    tempos_corte[c] = campo(s, 'tempo')
    print(f'corte={c:5d}  níveis de Strassen={int(math.log2(n // c)):2d}  tempo={tempos_corte[c]:.3f} s')

plt.figure(figsize=(6, 3.5))
plt.semilogx(cortes, [tempos_corte[c] for c in cortes], 'o-', base=2)
plt.axhline(tempos_corte[n], ls='--', color='gray', label='clássico (corte = n)')
plt.xlabel('ponto de corte $n_0$')
plt.ylabel('tempo (s)')
plt.title(f'Strassen híbrido, n = {n}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Curiosidade: Strassen como decomposição de tensor (a ideia do AlphaTensor)

Numere as entradas de $A$, $B$ e $C$ ($2\times 2$) como vetores de 4 posições:
$a = (A_{11}, A_{12}, A_{21}, A_{22})$, e da mesma forma $b$ e $c$. O produto de matrizes é
descrito por um **tensor** $\mathcal T$ de $4 \times 4 \times 4$ com
$\mathcal T[p, q, r] = 1$ quando o produto $a_p\, b_q$ entra em $c_r$, e $0$ caso contrário.

Uma receita com $R$ multiplicações é dada por três tabelas $U, V, W$ com $R$ colunas:

$$M_t = \Big(\sum_p U_{pt}\, a_p\Big)\Big(\sum_q V_{qt}\, b_q\Big), \qquad c_r = \sum_t W_{rt}\, M_t,$$

e ela está correta exatamente quando $\mathcal T = \sum_{t=1}^{R} U_{:,t} \otimes V_{:,t} \otimes W_{:,t}$.
O AlphaTensor (DeepMind, 2022) transformou a busca por $U, V, W$ com $R$ pequeno num jogo e
achou, por exemplo, uma receita $4 \times 4$ com 47 produtos em aritmética módulo 2. Em 2025, o
AlphaEvolve achou 48 para números complexos, contra os 49 de Strassen aplicado duas vezes.

**Fonte:** Fawzi et al. (2022), Fig. 1; Novikov et al. (2025).

In [ ]:
# Exemplo: o tensor do produto 2x2 e a decomposição de posto 7 de Strassen.
import numpy as np

def tensor_produto(m):
    """Tensor m^2 x m^2 x m^2 do produto de matrizes m x m (índices em ordem de linhas)."""
    T = np.zeros((m * m,) * 3, dtype=int)
    for i in range(m):
        for j in range(m):
            for k in range(m):           # a_ik * b_kj entra em c_ij
                T[i * m + k, k * m + j, i * m + j] = 1
    return T

def reconstroi(U, V, W):
    """Soma dos R produtos externos U[:,t] (x) V[:,t] (x) W[:,t]."""
    return np.einsum('pt,qt,rt->pqr', U, V, W)

# Colunas = M1..M7; linhas de U: A11 A12 A21 A22; de V: B11 B12 B21 B22; de W: C11 C12 C21 C22
U = np.array([[1, 0, 1, 0, 1, -1,  0],
              [0, 0, 0, 0, 1,  0,  1],
              [0, 1, 0, 0, 0,  1,  0],
              [1, 1, 0, 1, 0,  0, -1]])
V = np.array([[1, 1, 0, -1, 0, 1, 0],
              [0, 0, 1,  0, 0, 1, 0],
              [0, 0, 0,  1, 0, 0, 1],
              [1, 0, -1, 0, 1, 0, 1]])
W = np.array([[1,  0, 0, 1, -1, 0, 1],
              [0,  0, 1, 0,  1, 0, 0],
              [0,  1, 0, 1,  0, 0, 0],
              [1, -1, 1, 0,  0, 1, 0]])

T2 = tensor_produto(2)
print('Strassen reconstrói o tensor?', np.array_equal(reconstroi(U, V, W), T2),
      '| posto usado R =', U.shape[1])

# Usando a decomposição como algoritmo, no exemplo dos slides:
a = np.array([1, 3, 7, 5]); b = np.array([6, 8, 4, 2])
M = (U.T @ a) * (V.T @ b)            # as 7 multiplicações
print('M =', M, '-> C =', (W @ M).reshape(2, 2).tolist())

# Expoentes obtidos aplicando recursivamente uma receita base m x m com R produtos: log_m R
for nome, m, R in [('clássico 2x2', 2, 8), ('Strassen 2x2', 2, 7), ('Strassen 2 níveis 4x4', 4, 49),
                   ('AlphaEvolve 4x4 (complexos)', 4, 48), ('AlphaTensor 4x4 (mod 2)', 4, 47)]:
    print(f'{nome:30s} R={R:3d}  expoente = log_{m}({R}) = {math.log(R, m):.4f}')

In [ ]:
# Exercício 5: o algoritmo clássico também é uma decomposição (de posto 8)
# TODO: preencha U8, V8, W8 (4 linhas x 8 colunas) para as 8 multiplicações do
# algoritmo clássico 2x2, na ordem
#   A11*B11, A12*B21, A11*B12, A12*B22, A21*B11, A22*B21, A21*B12, A22*B22.
# Cada coluna de U8 e de V8 tem um único 1; cada coluna de W8 diz em qual C_ij o produto entra.

U8 = np.zeros((4, 8), dtype=int)   # TODO
V8 = np.zeros((4, 8), dtype=int)   # TODO
W8 = np.zeros((4, 8), dtype=int)   # TODO

# Teste automático
assert np.array_equal(reconstroi(U8, V8, W8), T2), 'Verifique sua implementação'
print('Exercício 5 ok! Clássico: posto 8. Strassen: posto 7. Winograd (1971): 7 é o mínimo.')

## Desafio Final

O programa `src/strassen.c` só aceita $n$ potência de 2. Escreva `src/strassen_geral.c`, que
multiplica matrizes de **qualquer** ordem $n$ e:

1. complete as matrizes com zeros (*padding*) até a próxima potência de 2 **ou**, melhor,
   trate $n$ ímpar removendo a última linha/coluna e corrigindo o resultado com produtos
   matriz-vetor (Cormen et al., 2009, Exercício 4.2-3);
2. use o ponto de corte que funcionou melhor na sua máquina na Seção 4;
3. confira o resultado contra o clássico para $n \in \{1, 2, 3, 5, 17, 100, 513\}$;
4. meça o tempo para $n = 600, 700, \dots, 1100$ e faça um gráfico comparando com o clássico.
   O *padding* gera "degraus" no gráfico. Explique onde eles aparecem e por quê.

**Bônus:** compare os seus tempos com `numpy` (`A @ B`), que chama uma BLAS otimizada.
Quantas vezes mais rápida ela é, e por quê, se a BLAS usa o algoritmo clássico?

In [ ]:
# Desafio: Strassen para n arbitrário
# TODO: resolva o desafio abaixo
# Sugestão: copie src/strassen.c para src/strassen_geral.c e comece pelo padding.

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §5.4.
- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. §4.2 e §4.5. (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- STRASSEN, V. Gaussian elimination is not optimal. *Numerische Mathematik*, v. 13, p. 354–356, 1969. DOI 10.1007/BF02165411.
- HIGHAM, N. J. *Accuracy and Stability of Numerical Algorithms*. 2. ed. SIAM, 2002. Cap. 23.
- FAWZI, A. et al. Discovering faster matrix multiplication algorithms with reinforcement learning. *Nature*, v. 610, p. 47–53, 2022. DOI 10.1038/s41586-022-05172-4.
- NOVIKOV, A. et al. AlphaEvolve: a coding agent for scientific and algorithmic discovery. arXiv:2506.13131, 2025.